# Backtest Framework Comparsion

This notebook calls the shared framework-comparison helper and only handles presentation.

It compares the same SMA crossover strategy across `backtesting.py`, Zipline Reloaded, and NautilusTrader on `yfinance` and `fmp` data, then decomposes whether vendor or framework differences dominate.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.backtests import run_framework_comparison

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

SYMBOL = 'AAPL'
PROVIDERS = ('yfinance', 'fmp')
FRAMEWORKS = ('backtesting.py', 'zipline', 'nautilus')
START = '2020-01-01'
END = None
FAST_WINDOW = 50
SLOW_WINDOW = 200
CAPITAL_BASE = 100_000.0


In [2]:
result = run_framework_comparison(
    symbol=SYMBOL,
    providers=PROVIDERS,
    frameworks=FRAMEWORKS,
    start=START,
    end=END,
    fast_window=FAST_WINDOW,
    slow_window=SLOW_WINDOW,
    capital_base=CAPITAL_BASE,
)

display(result.comparison.loc[:, [
    'provider', 'framework', 'symbol', 'bars', 'trades', 'final_equity',
    'total_return', 'max_drawdown', 'performance_score', 'elapsed_seconds', 'wall_clock_seconds', 'bars_per_second',
    ]].round(4))
display(result.factor_report.round(4))
display(result.framework_summary.round(4))
display(result.provider_summary.round(4))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/backtesting/_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


,provider,framework,symbol,bars,trades,final_equity,total_return,max_drawdown,performance_score,elapsed_seconds,wall_clock_seconds,bars_per_second
0,fmp,backtesting.py,AAPL,1627,5,120703.45,0.2070,-0.2120,-0.0050,0.0173,0.0295,94214.12
1,fmp,nautilus,AAPL,1627,9,119306.20,0.1931,-0.2102,-0.0171,0.0943,0.8183,17262.31
2,fmp,zipline,AAPL,1428,9,113284.02,0.1328,-0.2011,-0.0683,0.5819,0.5842,2454.24
3,yfinance,backtesting.py,AAPL,1626,5,121619.84,0.2162,-0.2050,0.0112,0.0178,0.2815,91438.37
4,yfinance,nautilus,AAPL,1626,9,116616.60,0.1662,-0.2089,-0.0427,0.1028,0.8433,15824.29
5,yfinance,zipline,AAPL,1427,9,115719.97,0.1572,-0.1994,-0.0422,0.6666,0.9313,2140.78


,metric,framework_ss,provider_ss,residual_ss,framework_share,provider_share,residual_share,dominant_factor,provider_to_framework_ratio
0,performance_score,0.0034,0.0000,0.0008,0.8107,0.0110,0.1783,framework,0.0136
1,total_return,0.0044,0.0000,0.0007,0.8635,0.0015,0.1351,framework,0.0017
2,elapsed_seconds,0.4340,0.0015,0.0022,0.9917,0.0033,0.0049,framework,0.0034
3,wall_clock_seconds,0.5494,0.0649,0.0274,0.8562,0.1012,0.0426,framework,0.1182


,mean_total_return,mean_max_drawdown,mean_elapsed_seconds,mean_wall_clock_seconds,mean_bars_per_second
framework,,,,,
backtesting.py,0.2116,-0.2085,0.0176,0.1555,92826.245
nautilus,0.1796,-0.2096,0.0986,0.8308,16543.300
zipline,0.1450,-0.2002,0.6242,0.7578,2297.510


,mean_total_return,mean_max_drawdown,mean_elapsed_seconds,mean_wall_clock_seconds,mean_bars_per_second
provider,,,,,
fmp,0.1776,-0.2078,0.2312,0.4773,37976.8900
yfinance,0.1799,-0.2044,0.2624,0.6854,36467.8133


The first table is the direct comparison. The second table is the comparison-of-comparisons: if `provider_ss` is larger than `framework_ss`, the vendor choice moved the strategy more than the engine choice for this setup.